In [ ]:
%xmode Context
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt

import sys
from pathlib import Path

# help locating the package sgpykit (comment this out if you installed it)
PKG_PARENT = Path().resolve().parent
sys.path.insert(0, str(PKG_PARENT))

import sgpykit as sg

In [ ]:
import logging
from sgpykit.util.log import logger

logging.basicConfig(
    # see https://docs.python.org/3/library/logging.html#logrecord-attributes
    format="%(levelname)s %(name)s:%(lineno)d: %(message)s",
)
# switching log level for sgpykit. Also see https://docs.python.org/3/library/logging.html#logging-levels
# logger.setLevel(logging.INFO)
logger.setLevel(logging.DEBUG)

In [ ]:
# function to integrate / interpolate
f = lambda x: 1.0/(x[0]**2 + x[1]**2 + 0.3)

N = 2
a, b = -1.0, 1.0

# knot generator (Clenshaw-Curtis on [a,b])
knots = lambda n: sg.knots_CC(n, a, b)

# level-to-knots mapping (doubling rule)
lev2knots = sg.lev2knots_doubling

controls = {
    "max_pts": 20,
    "prof_tol": 1e-10,
    "nested": True,
    "plot": False
}
prev_adapt = None

adapt1 = sg.adapt_sparse_grid(f, N, knots, lev2knots, prev_adapt, controls)

In [ ]:
sg.plot_idx_status(adapt1.private["G"],
                   adapt1.private["I"],
                   adapt1.private["idx_bin"],
                   adapt1.private["idx"])

In [ ]:
# ------------------------------------------------------------
# 2-D example - custom profit indicator
# ------------------------------------------------------------
f = lambda x: 1.0/(x[0]**2 + x[1]**2 + 0.3)

controls.update({
    "prof": "deltaint/new_points",   # custom profit
    "nested": True
})
prev_adapt = None
adapt2 = sg.adapt_sparse_grid(f, N, knots, lev2knots, prev_adapt, controls)

In [ ]:
sg.plot_idx_status(adapt2.private["G"],
                   adapt2.private["I"],
                   adapt2.private["idx_bin"],
                   adapt2.private["idx"])

In [ ]:
# ------------------------------------------------------------
# Increase the number of points - recycle previous run
# ------------------------------------------------------------
controls["max_pts"] = 1500
logger.setLevel(logging.INFO)
adapt3 = sg.adapt_sparse_grid(f, N, knots, lev2knots, adapt2, controls)

In [ ]:
sg.plot_idx_status(adapt3.private["G"],
                   adapt3.private["I"],
                   adapt3.private["idx_bin"],
                   adapt3.private["idx"])

In [ ]:
from scipy.stats import norm
# ------------------------------------------------------------
# 2-D example on an unbounded interval (Gaussian weight)
# ------------------------------------------------------------
f = lambda x: 1.0/(2 + np.exp(x[0]) + np.exp(x[1]))
N = 2

knots = lambda n: sg.knots_GK(n, 0, 1)          # Gauss-Kronrod on [0,1]
lev2knots = sg.lev2knots_GK

controls = {
    "max_pts": 150,
    "prof_tol": 1e-10,
    "prof": "weighted Linf/new_points",
    "nested": True,
    # Gaussian pdf (product of 1-D normals)
    "pdf": lambda Y: np.prod(norm.pdf(Y, 0, 1), axis=0)
}
prev_adapt = None
adapt1 = sg.adapt_sparse_grid(f, N, knots, lev2knots, prev_adapt, controls)

In [ ]:
sg.plot_idx_status(adapt1.private["G"],
                   adapt1.private["I"],
                   adapt1.private["idx_bin"],
                   adapt1.private["idx"])

In [ ]:
# ---- non-nested Gaussian knots ---------------------------------
knots = lambda n: sg.knots_normal(n, 0, 1)     # Gauss-Hermite (non-nested)
lev2knots = sg.lev2knots_lin
controls = {
        "max_pts": 150,
        "prof_tol": 1e-10,
        "prof": "weighted Linf/new_points",
        "nested": False,
        "plot": False,
        # Gaussian pdf (product of 1-D normals)
        "pdf": lambda Y: np.prod(norm.pdf(Y, 0, 1), axis=0)
    }
# here's the adapt non-nested. You will see some message like:
# "Some points have been evaluated more than once. Total: 191 extra evaluations over 295 function evaluations"
# you can change this behaviour by changing the default value of controls.recycling from 
# controls.recycling = 'priority_to_evaluation'
# to
# controls.recycling = 'priority_to_recycling'
# however, this is **not** recommended if N is large and evaluating f is cheap.
# see help ADAPT_SPARSE_GRID > CONTROLS.RECYCLING for more information
adapt2 = sg.adapt_sparse_grid(f, N, knots, lev2knots, prev_adapt, controls)

In [ ]:
sg.plot_idx_status(adapt2.private["G"],
                   adapt2.private["I"],
                   adapt2.private["idx_bin"],
                   adapt2.private["idx"])
print("Integral (nested):   ", adapt1.intf)
print("Integral (non-nested):", adapt2.intf)

In [ ]:
# change recycling priority (optional)
controls["recycling"] = "priority_to_recycling"
adapt3 = sg.adapt_sparse_grid(f, N, knots, lev2knots, prev_adapt, controls)
# this will be false, because num_evals and nb_pts_visited are now identical
print("Same result after changing recycling? ", adapt2 == adapt3)
# observe that there's a maximum number of tabulated points with GK that one will hit sooner or later, so asking too many points
# will result in an error

In [ ]:
# ------------------------------------------------------------
# Too many points - error handling (Gauss-Kronrod)
# ------------------------------------------------------------
f = lambda x: 1.0/(2 + np.exp(x[0]) + np.exp(x[1]))
N = 2
knots = lambda n: sg.knots_GK(n, 0, 1)
lev2knots = sg.lev2knots_GK

controls = {
    "max_pts": 300,
    "prof_tol": 1e-10,
    "pdf": lambda Y: np.exp(-0.5*np.sum(Y**2, axis=0)),   # weight for profit
    "prof": "weighted Linf/new_points",
    "nested": True
}
prev_adapt = None

try:
    adapt1 = sg.adapt_sparse_grid(f, N, knots, lev2knots, prev_adapt, controls)
except Exception as e:
    print("-" * 30)
    print("We are asking too many points (level beyond 5).")
    print(f"lev2knots_GK would return Inf => error: {e}")
    print("Reduce controls['max_pts'] (e.g. to 300).")
    print("-" * 30)

In [ ]:
# ------------------------------------------------------------
# Non-weighted profit - may also raise an error
# ------------------------------------------------------------
controls = {
    "max_pts": 1500,
    "prof": "Linf/new_points",
    "prof_tol": 1e-10,
    "nested": True
}
try:
    adapt1 = sg.adapt_sparse_grid(f, N, knots, lev2knots, None, controls)
except Exception as e:
    print("-" * 30)
    print("We are asking too many points (level beyond 5).")
    print(f"lev2knots_GK would return Inf → error: {e}")
    print("Reduce controls['max_pts'] (e.g. to 300).")
    print("-" * 30)

In [ ]:
# ------------------------------------------------------------
# Vector-valued function (2 components)
# ------------------------------------------------------------
f = lambda x: np.array([
    1.0/(x[0]**2 + x[1]**2 + 0.3),
    1.0/(x[0]**2 + 0.1*x[1]**2 + 2)
])

N = 2
a, b = -1.0, 1.0
knots = lambda n: sg.knots_CC(n, a, b)
lev2knots = sg.lev2knots_doubling

controls = {
    "max_pts": 200,
    "prof_tol": 1e-10,
    "nested": True,
    "plot": False
}
prev_adapt = None
adapt1 = sg.adapt_sparse_grid(f, N, knots, lev2knots, prev_adapt, controls)

In [ ]:
# adaptivity for vector-valued quantites is done by computing by default the euclidean norm of the output.
# by changing the way in which profit of vector valued quantities are computed, I can recover exactly the same
# behaviour as if I was considering only one of the two components
# ---- work on a single component only -------------------------
# use e.g. this one to recover the scalar result for the first function only
controls["op_vect"] = lambda A, B: np.sqrt(np.sum((A[0, :] - B[0, :])**2, axis=0))
adapt_f1 = sg.adapt_sparse_grid(f, N, knots, lev2knots, None, controls)

In [ ]:
# compare with scalar version
f1 = lambda x: 1.0/(x[0]**2 + x[1]**2 + 0.3)
controls.pop("op_vect", None)   # restore default
adapt_f1_check = sg.adapt_sparse_grid(f1, N, knots, lev2knots, None, controls)

In [ ]:
# use e.g. this one to recover the scalar result for the second function only
controls["op_vect"] = lambda A, B: np.sqrt(np.sum((A[1, :] - B[1, :])**2, axis=0))
adapt_f2 = sg.adapt_sparse_grid(f, N, knots, lev2knots, None, controls)

In [ ]:
f2 = lambda x: 1.0/(x[0]**2 + 0.1*x[1]**2 + 2)
controls.pop("op_vect", None)   # restore default
adapt_f2_check = sg.adapt_sparse_grid(f2, N, knots, lev2knots, None, controls)

In [ ]:
# ------------------------------------------------------------
# 4-D example
# ------------------------------------------------------------
f = lambda x: 1.0/(x[0]**2 + x[1]**2 + 0.3 + 0.1*np.sin(x[2])*np.exp(0.4*x[3]))
N = 4
a, b = -1.0, 1.0
knots = lambda n: sg.knots_CC(n, a, b)
lev2knots = sg.lev2knots_doubling

controls = {
    "max_pts": 400,
    "prof_tol": 1e-10,
    "nested": True
}
prev_adapt = None

In [ ]:
import time
logger.setLevel(logging.INFO)
t0 = time.time()
adapt1 = sg.adapt_sparse_grid(f, N, knots, lev2knots, prev_adapt, controls)
print(f"4-D adapt time: {time.time() - t0:.2f}s")

In [ ]:
# Plot 2-D projections of the multi-index set (optional)
# plot indices. PS those idx_bin in the corner next to the origin are 2D projection of 4D indices not in idx_bin, like
# [1 1 4 1], so that's ok
sg.plot_idx_status(adapt1.private["G"][:, :2],
                   adapt1.private["I"][:, :2],
                   adapt1.private["idx_bin"][:, :2],
                   adapt1.private["idx"][:2])
sg.plot_idx_status(adapt1.private["G"][:, 2:4],
                   adapt1.private["I"][:, 2:4],
                   adapt1.private["idx_bin"][:, 2:4],
                   adapt1.private["idx"][2:4])

In [ ]:
# now we verify that using partial exploration we gain in computational work
# ------------------------------------------------------------
# Partial-dimension exploration (buffer) - 25-D toy problem
# ------------------------------------------------------------
#  the target function is a 25-dim function, only the first 3 are meaningful:
#  f=@(y) 1./(4 + y(1) + 0.2*y(2) + 0.04*y(3) + 0*y(4:25) );
# 
#  so with a buffer of 2 we expect that dimadapt will explore only the first 5, with a gain
#  of 20x2= 40 points (20 unexplored dim x 2 points to explore each of them to realize that they 
#  are meaningless)
# 
#  Because f must be ablo to accept multiple dimensions of y, so we hack it as:
aaa = np.array([1, 0.2, 0.04] + [0]*22)   # length 25
f = lambda y: 1.0/(4 + np.dot(aaa[:len(y)], y))

N = len(aaa)
a, b = -1.0, 1.0
knots = lambda n: sg.knots_CC(n, a, b)
lev2knots = sg.lev2knots_doubling

controls = {
    "prof_tol": 1e-10,
    "nested": True
}
prev_adapt = None

# ---- standard adapt (convergence study) -----------------------
# we do a convergence study for standard adapt, with 30 values of work between 0 and 300. The reference is set
# at 500 points
nb_pts = 30
max_pts = np.ceil(np.logspace(1, np.log10(300), nb_pts)).astype(int).tolist() + [500]

intf_vals = []
true_pts = []

logger.setLevel(logging.INFO)
for p in max_pts:
    controls["max_pts"] = p
    adapt = sg.adapt_sparse_grid(f, N, knots, lev2knots, prev_adapt, controls)
    intf_vals.append(adapt['intf'])
    true_pts.append(adapt['nb_pts'])
    if p < max_pts[-1]:
        prev_adapt = adapt

In [ ]:
# reference solution (last run)
adapt_ref = prev_adapt
intf_ex = intf_vals[-1]
intf_vals = intf_vals[:-1]
max_pts = max_pts[:-1]
true_pts = true_pts[:-1]

# ---- dimension-adapt (buffer) ---------------------------------
controls["var_buffer_size"] = 2
N_full = N
dimad_intf_vals = []
dimad_true_pts = []
dimad_N = []

logger.setLevel(logging.INFO)
prev_adapt = None
for p in max_pts:
    controls["max_pts"] = p
    adapt = sg.adapt_sparse_grid(f, N_full, knots, lev2knots, prev_adapt, controls)
    dimad_intf_vals.append(adapt['intf'])
    dimad_true_pts.append(adapt['nb_pts'])
    dimad_N.append(adapt['N'])
    prev_adapt = adapt

In [ ]:
# ---- Plot convergence (TODO) ------------------------------
# we now plot the convergence. The 40 points gain is confirmed
# plt.figure()
# plt.loglog(true_pts, np.abs(np.array(intf_vals)-intf_ex), '-ob',
#            linewidth=2, markerfacecolor='b', label='standard adapt')
# plt.loglog(dimad_true_pts, np.abs(np.array(dimad_intf_vals)-intf_ex), '-or',
#            linewidth=2, markerfacecolor='r', label='dim-adapt')
# plt.legend()
# plt.grid(True)
# plt.show()
# % we also plot the sequence with which indices are added to the grid
# figure
# subplot(1,2,1)
# spy(adapt1.private.G_log-1)
# subplot(1,2,2)
# spy(adapt2.private.G_log-1)
 
# % observe that after the initial part in which dimadapt gains its advantage, they then continue in the same order. 
# figure
# subplot(1,2,1)
# spy(adapt1.private.G_log(30:end,1:3)-1)
# subplot(1,2,2)
# spy(adapt2.private.G_log(10:end,1:3)-1)

# % the same can be seen from the pts count, which is parallel, 40 pts diff
# figure
# plot(adapt1.private.nb_pts_log(30:end),'b','DisplayName','adapt')
# hold on
# plot(adapt2.private.nb_pts_log(10:end),'r','DisplayName','dim adapt')
# legend show

# ------------------------------------------------------------
# Non-nested uniform knots - same experiment with buffer
# ------------------------------------------------------------
#  something similar happens also for non-nested points. Dspite the linear growth, using non-nested points requires 2 new points to assess that
#  a dimension in useless (2 pts at level 2, different from the point at level 1) so the gain is still 20x2 =
#  40 points. Things are less evident though, because nb_pts_log reports the count of Tr and not of unique(Hr),
#  even if I set priority to recycling the lines won't be parallel
aaa = np.array([1, 0.2, 0.04] + [0]*22)
f = lambda y: 1.0/(4 + np.dot(aaa[:len(y)], y))

N = len(aaa)
a, b = -1.0, 1.0
knots = lambda n: sg.knots_uniform(n, a, b)
lev2knots = sg.lev2knots_lin

controls = {
    "prof_tol": 1e-10,
    "nested": False
}

prev_adapt = None
logger.setLevel(logging.INFO)

# ---- standard adapt -------------------------------------------
# we do a convergence study for standard adapt, with 30 values of work between 0 and 300. The reference is set
# at 500 points
intf_vals = []
true_pts = []
for p in max_pts:
    controls["max_pts"] = p
    adapt = sg.adapt_sparse_grid(f, N, knots, lev2knots, prev_adapt, controls)
    intf_vals.append(adapt['intf'])
    true_pts.append(adapt['nb_pts'])
    if p < max_pts[-1]:
        prev_adapt = adapt

In [ ]:
# move the last solve to reference and delete it from the convergence study
adapt_ref = prev_adapt
intf_ex = intf_vals[-1]
intf_vals = intf_vals[:-1]
true_pts = true_pts[:-1]

# ---- dim-adapt with buffer --------------------------------------
controls["var_buffer_size"] = 2
dimad_intf_vals = []
dimad_true_pts = []
dimad_N = []

prev_adapt = None
logger.setLevel(logging.INFO)

for p in max_pts:
    controls["max_pts"] = p
    adapt = sg.adapt_sparse_grid(f, N, knots, lev2knots, prev_adapt, controls)
    dimad_intf_vals.append(adapt['intf'])
    dimad_true_pts.append(adapt['nb_pts'])
    dimad_N.append(adapt['N'])
    prev_adapt = adapt

# (TODO)
# %% we now plot the convergence. The 40 points gain is confirmed
# figure
# loglog(true_pts,abs(intf_vals-intf_ex),'-ob','LineWidth',2,'MarkerFaceColor','b','DisplayName','standard adapt')
# hold on
# loglog(dimad_true_pts,abs(dimad_intf_vals-intf_ex),'-or','LineWidth',2,'MarkerFaceColor','r','DisplayName','dim-adapt')
# legend show
# grid on

# % we also plot the sequence with which indices are added to the grid
# figure
# subplot(1,2,1)
# spy(adapt1.private.G_log-1)
# subplot(1,2,2)
# spy(adapt2.private.G_log-1)
 
# % observe that after the initial part in which dimadapt gains its advantage, they then continue in the same order. 

# figure
# subplot(1,2,1)
# spy(adapt1.private.G_log(31:end,1:3)-1)
# subplot(1,2,2)
# spy(adapt2.private.G_log(11:end,1:3)-1)

# % the same can be seen from the pts count, which is parallel, 40 pts diff
# figure
# plot(adapt1.private.nb_pts_log(31:end),'b','DisplayName','adapt')
# hold on
# plot(adapt2.private.nb_pts_log(11:end),'r','DisplayName','dim adapt')
# legend show

In [ ]:
# ------------------------------------------------------------
# Convergence study in L-inf norm (max interpolation error)
# ------------------------------------------------------------
f = lambda x: 1.0/(x[0]**2 + x[1]**2 + 0.3)

N = 2
a, b = -1.0, 1.0
knots = lambda n: sg.knots_CC(n, a, b)
lev2knots = sg.lev2knots_doubling

controls = {
    "prof_tol": 1e-10,
    "nested": True
}
prev_adapt = None
logger.setLevel(logging.INFO)

# random test points for error evaluation
# evaluate error as max error over 100 random points in [a,b]^2. Note that here we have hard-coded that a=-1,  b=1
nb_rand_pts = 100
Rand_pts = 2*np.random.rand(2, nb_rand_pts) - 1   # shape (2,100)
truef_evals = f(Rand_pts)
# generate a sequence of sparse grids with these many points (approximately), for each save values of interest 
# (exact nb pts, error,  approximation of integral of f)
max_pts = [5, 7, 13, 21, 29, 50, 80, 200, 400, 600, 1000]
sg_pts = []
sg_err = []
quadf_vals = []

for p in max_pts:
    controls["max_pts"] = p
    adapt = sg.adapt_sparse_grid(f, N, knots, lev2knots, prev_adapt, controls)
    # interpolate on the current sparse grid
    sg_eval = sg.interpolate_on_sparse_grid(adapt['S'], adapt['Sr'],
                                            adapt['f_on_Sr'], Rand_pts)
    sg_err.append(np.max(np.abs(sg_eval - truef_evals)))
    quadf_vals.append(adapt['intf'])
    sg_pts.append(adapt['nb_pts'])
    if p < max_pts[-1]:
        prev_adapt = adapt

In [ ]:
# reference quadrature value (last computed)
quadf_ref = quadf_vals[-1]
quadf_vals = quadf_vals[:-1]
sg_pts = sg_pts[:-1]

# ---- Plot quadrature error (TODO) ------------------------------------
# plt.figure()
# plt.loglog(sg_pts, np.abs(np.array(quadf_vals)-quadf_ref), '-ob',
#            linewidth=2, markerfacecolor='b', label='adaptive sg')
# plt.title('quadrature error')
# plt.grid(True)
# plt.show()

# ---- Plot interpolation error ---------------------------------
# plt.figure()
# plt.loglog(sg_pts, sg_err, '-ob',
#            linewidth=2, markerfacecolor='b', label='adaptive sg')
# plt.title('interp error (L∞)')
# plt.grid(True)
# plt.show()


# ------------------------------------------------------------
# Different knots per dimension (mixed intervals)
# ------------------------------------------------------------
f = lambda x: 1.0/(x[0]**2 + x[1]**2 + 2)
N = 2

controls = {
    "max_pts": 200,
    "prof_tol": 1e-10,
    "plot": False
}
prev_adapt = None

# ---- nested case (different intervals) -----------------------
knots = [
    lambda n: sg.knots_CC(n, 0, 1),   # dim-1 on [0,1]
    lambda n: sg.knots_CC(n, 3, 5)    # dim-2 on [3,5]
]
lev2knots = sg.lev2knots_doubling
controls["nested"] = True

adapt = sg.adapt_sparse_grid(f, N, knots, lev2knots, prev_adapt, controls)
print("Integral (mixed intervals, nested):", adapt['intf'])

In [ ]:
# build the sparse grid explicitly (optional)
G = adapt['private']['G']
S,_ = sg.create_sparse_grid_multiidx_set(G, knots, lev2knots)
Sr = sg.reduce_sparse_grid(S)
Q1 = sg.quadrature_on_sparse_grid(f, S=None, Sr=Sr)

# alternative grid (full tensor up to level 8)
S2,_ = sg.create_sparse_grid_multiidx_set(sg.fast_TD_set(N, 8), knots, lev2knots)
Sr2 = sg.reduce_sparse_grid(S2)
Q2 = sg.quadrature_on_sparse_grid(f, S=None, Sr=Sr2)

In [ ]:
# more examples on using the buffer, to check that in some setups it won't work. To do this,  we compare results with and without buffer,  which will be different
# ------------------------------------------------------------
# Buffer tests - 3-D examples with various intervals
# ------------------------------------------------------------
import scipy.stats as stats   # for normal pdf if needed

logger.setLevel(logging.INFO)

for test_case in range(1, 6):
    # -----------------------------------------------------------------
    # define the test case
    # -----------------------------------------------------------------
    if test_case == 1:          # works
        f = lambda x: 1.0/np.exp(np.sum(x, axis=0))
        knots_f = [
            lambda n: sg.knots_CC(n, -0.5, 0.5),
            lambda n: sg.knots_CC(n, -0.5, 0.5),
            lambda n: sg.knots_CC(n, -0.2, 0.2)
        ]
    elif test_case == 2:        # does NOT work
        f = lambda x: 1.0/np.exp(np.sum(x, axis=0))
        knots_f = [lambda n: sg.knots_CC(n, 0, 1)]*3
    elif test_case == 3:        # does NOT work
        f = lambda x: np.prod(x, axis=0)
        knots_f = [lambda n: sg.knots_CC(n, 0, 1)]*3
    elif test_case == 4:        # does NOT work
        f = lambda x: np.sum(np.cos(x), axis=0)
        knots_f = [lambda n: sg.knots_CC(n, -1, 1)]*3
    elif test_case == 5:        # works
        f = lambda x: np.prod(x, axis=0)
        knots_f = [lambda n: sg.knots_CC(n, 0, 2)]*3

    N = 3
    lev2knots = sg.lev2knots_doubling

    controls = {
        "max_pts": 200,
        "prof_tol": 1e-10,
        "plot": False,
        "var_buffer_size": 2,          # buffer version
        "nested": True
    }
    prev_adapt = None

    # ---- with buffer ------------------------------------------------
    adapt_buff = sg.adapt_sparse_grid(f, N, knots_f, lev2knots,
                                     prev_adapt, controls)

    # ---- without buffer (var_buffer_size = N) -----------------------
    controls["var_buffer_size"] = N
    adapt_no_buff = sg.adapt_sparse_grid(f, N, knots_f, lev2knots,
                                         prev_adapt, controls)

    # ---- one-shot construction (no adapt) ---------------------------
    G = adapt_no_buff['private']['G']
    S,_ = sg.create_sparse_grid_multiidx_set(G, knots_f, lev2knots)
    Sr = sg.reduce_sparse_grid(S)
    Q,_ = sg.quadrature_on_sparse_grid(f, S=None, Sr=Sr)

    print("\n------------")
    print(f"test case: {test_case}")
    print(f"with buffer integral:   {adapt_buff['intf'][0]}")
    print(f"no buffer integral:    {adapt_no_buff['intf'][0]}")
    print(f"one-shot integral:     {Q[0]}")